# Pydantic Structured Output Examples
One cell per provider/model setup from benchmark cases.

In [18]:
import patch
from pydantic import BaseModel
from release import Provider
from release.providers.base import ModelMetadata
from release.providers.openai import OpenAIProvider
from release.providers.google import GoogleProvider
from release.providers.anthropic import AnthropicProvider
from release.providers.openrouter import OpenRouterProvider

In [8]:
class LessonSummary(BaseModel):
    topic: str
    key_points: list[str]
    difficulty: str

In [9]:
async def run_pydantic_case(name: str, provider: Provider, model: str, model_metadata: list[ModelMetadata]) -> None:
    stream = await Provider.PydanticStream.Select(
        Provider.PydanticStream.Request(
            model=model,
            model_metadata=model_metadata,
            type=LessonSummary,
            messages=[
                {'role': 'system', 'content': 'You produce compact educational summaries.'},
                {'role': 'user', 'content': 'Return a lesson summary about latency benchmarking with topic, 3 key_points, and difficulty.'},
            ],
        ),
        provider,
    )
    final_model = None
    print(f'[{name}] model={model}')
    async for event in stream:
        if event.type == 'error':
            print(f'[Error: {event.payload}]')
            return
        final_model = event.payload.current
        print(f'partial[{event.payload.i}]: {final_model.model_dump()}')
    if final_model is not None:
        print('final:')
        print(final_model.model_dump_json(indent=2))

In [10]:
await run_pydantic_case('openai 4o', OpenAIProvider(), 'gpt-4o', [OpenAIProvider.ModelMetadata()])

[openai 4o] model=gpt-4o
partial[0]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[1]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[2]: {'topic': '', 'key_points': None, 'difficulty': None}
partial[3]: {'topic': 'Latency', 'key_points': None, 'difficulty': None}
partial[4]: {'topic': 'Latency Benchmark', 'key_points': None, 'difficulty': None}
partial[5]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[6]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[7]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[8]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[9]: {'topic': 'Latency Benchmarking', 'key_points': [''], 'difficulty': None}
partial[10]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency'], 'difficulty': None}
partial[11]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency benchmarking'], 'diffi

In [11]:
await run_pydantic_case('openai 5.2', OpenAIProvider(), 'gpt-5.2-2025-12-11', [OpenAIProvider.ModelMetadata()])

[openai 5.2] model=gpt-5.2-2025-12-11
partial[0]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[1]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[2]: {'topic': '', 'key_points': None, 'difficulty': None}
partial[3]: {'topic': 'Latency', 'key_points': None, 'difficulty': None}
partial[4]: {'topic': 'Latency benchmarking', 'key_points': None, 'difficulty': None}
partial[5]: {'topic': 'Latency benchmarking', 'key_points': None, 'difficulty': None}
partial[6]: {'topic': 'Latency benchmarking', 'key_points': None, 'difficulty': None}
partial[7]: {'topic': 'Latency benchmarking', 'key_points': None, 'difficulty': None}
partial[8]: {'topic': 'Latency benchmarking', 'key_points': [''], 'difficulty': None}
partial[9]: {'topic': 'Latency benchmarking', 'key_points': ['Define'], 'difficulty': None}
partial[10]: {'topic': 'Latency benchmarking', 'key_points': ['Define clear'], 'difficulty': None}
partial[11]: {'topic': 'Latency benchmarking', 'key_points': ['Def

In [17]:
await run_pydantic_case('google', GoogleProvider(), 'gemini-3-flash-preview', [GoogleProvider.ModelMetadata(thinking_level='minimal')])

[google] model=gemini-3-flash-preview
partial[0]: {'topic': 'Latency Benchmarking', 'key_points': ['Measures the time delay between a request and a response, focusing on metrics like Round-Trip Time (RTT) and Time to First Byte (TTFB).', "Focuses on distribution percentiles (P95, P99) rather than averages to identify 'tail latency' issues that affect the slowest user experiences.", 'Requires controlled environments to minimize noise from background processes, network fluctuations, and hardware thermal throttling.'], 'difficulty': 'Intermediate'}
final:
{
  "topic": "Latency Benchmarking",
  "key_points": [
    "Measures the time delay between a request and a response, focusing on metrics like Round-Trip Time (RTT) and Time to First Byte (TTFB).",
    "Focuses on distribution percentiles (P95, P99) rather than averages to identify 'tail latency' issues that affect the slowest user experiences.",
    "Requires controlled environments to minimize noise from background processes, network f

In [15]:
await run_pydantic_case('anthropic', AnthropicProvider(), 'claude-sonnet-4-5-20250929', [AnthropicProvider.ModelMetadata()])

[anthropic] model=claude-sonnet-4-5-20250929
partial[0]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[1]: {'topic': None, 'key_points': None, 'difficulty': None}
partial[2]: {'topic': 'Latenc', 'key_points': None, 'difficulty': None}
partial[3]: {'topic': 'Latency Benchma', 'key_points': None, 'difficulty': None}
partial[4]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[5]: {'topic': 'Latency Benchmarking', 'key_points': None, 'difficulty': None}
partial[6]: {'topic': 'Latency Benchmarking', 'key_points': [], 'difficulty': None}
partial[7]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency mea'], 'difficulty': None}
partial[8]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency measures th'], 'difficulty': None}
partial[9]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency measures the time'], 'difficulty': None}
partial[10]: {'topic': 'Latency Benchmarking', 'key_points': ['Latency measures the time dela'], 

In [16]:
await run_pydantic_case('openrouter', OpenRouterProvider(), 'moonshotai/kimi-k2.5', [OpenRouterProvider.ModelMetadata(reasoning_enabled=False)])

[openrouter] model=moonshotai/kimi-k2.5
partial[0]: {'topic': 'Latency', 'key_points': None, 'difficulty': None}
partial[1]: {'topic': 'Latency Benchmarking in', 'key_points': None, 'difficulty': None}
partial[2]: {'topic': 'Latency Benchmarking in Distributed', 'key_points': None, 'difficulty': None}
partial[3]: {'topic': 'Latency Benchmarking in Distributed Systems', 'key_points': None, 'difficulty': None}
partial[4]: {'topic': 'Latency Benchmarking in Distributed Systems', 'key_points': ['Latency'], 'difficulty': None}
partial[5]: {'topic': 'Latency Benchmarking in Distributed Systems', 'key_points': ['Latency measures the'], 'difficulty': None}
partial[6]: {'topic': 'Latency Benchmarking in Distributed Systems', 'key_points': ['Latency measures the time taken for'], 'difficulty': None}
partial[7]: {'topic': 'Latency Benchmarking in Distributed Systems', 'key_points': ['Latency measures the time taken for a request to travel'], 'difficulty': None}
partial[8]: {'topic': 'Latency Benc